# Paper 2 follow-up experiments: Kaggle T4 x2 (cell F: ResNet-18 without BN; cell E: LR x decay grid)

This notebook runs in **two stages**, selected with `STAGE` in Cell 1.

**Stage 1, `STAGE = 'pilot'` (interactive, ~30 min):** checks that ResNet-18 **without BN** trains at all.
- **E7p**: Adam, LR 1e-4 / 3e-4 / 1e-3, 30 epochs, on cell **F** (plain ResNet-18 without BN) and cell **H** (fallback: zero-initialised scalar on each residual branch).
- Cell 7 prints a table and **recommends F or H**. (In the reported study, F trained well and was used; H was not needed.)

**Stage 2, `STAGE = 'main'` (Save & Run All (Commit), ~11 h):**
- **E7**: the six configurations (Adam, AdamW, SF-AdamW with L2, SF-AdamW, AdaBelief, AdaBelief with decoupled decay) on cell F: 30-epoch LR search, then 5 seeds x 100 epochs. (If the pilot recommends H, set `USE_H = True` to run **E7h** on cell H.)
- **E8** (cell E): the missing points of the 6 LR x 6 decay-coefficient grid for Adam and AdamW, 1 seed, 50 epochs.

**Settings (right panel):** Accelerator `GPU T4 x2` · Internet `On` · Persistence `Files only`.

**Input:** (1) a Kaggle dataset with this repository's code (e.g. a zip of the repo named `adam-bench-v5`); (2) optionally a CIFAR-10 dataset; (3) to resume an interrupted session, add this notebook's own Output as Input.


In [ ]:
# === Cell 1. Parameters ===
STAGE  = 'pilot'   # 'pilot' (stage 1, interactive) or 'main' (stage 2, Commit)
USE_H  = False     # set True only if Cell 7 of the pilot recommends H
if STAGE == 'pilot':
    CELLS, EXPS = ['F', 'H'], ['E7p']
else:
    CELLS, EXPS = (['H', 'E'], ['E7h', 'E8']) if USE_H else (['F', 'E'], ['E7', 'E8'])
RUN_CHECKS    = False    # True once to run the self-test (~5 min), then back to False
MAX_HOURS     = 10.5     # no new run is started after this many hours
TIME_BUDGET_H = 11.25    # hard stop (Kaggle ends sessions at ~12 h)
N_WORKERS     = 2

WORK    = '/kaggle/working'
CODE    = f'{WORK}/adam-bench-v5'
DATA    = f'{WORK}/data'
RESULTS = f'{WORK}/results_exp'


In [ ]:
# === Cell 2. Copy code and data (downloaded once so the two workers do not collide); restore results of a previous session ===
import glob, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

runner = [h for h in glob.glob('/kaggle/input/**/run_exp.py', recursive=True) if '/results' not in h]
if not runner:
    raise SystemExit('run_exp.py not found in Input: add the dataset with the code (adam-bench-v5).')
if os.path.exists(CODE):
    shutil.rmtree(CODE)
shutil.copytree(os.path.dirname(runner[0]), CODE,
                ignore=shutil.ignore_patterns('results*', 'data', '__pycache__'))
print('Code:', os.path.dirname(runner[0]))

os.makedirs(DATA, exist_ok=True)
dst = os.path.join(DATA, 'cifar-10-batches-py')
if not os.path.exists(dst):
    dirs = [d for d in glob.glob('/kaggle/input/**/cifar-10-batches-py', recursive=True) if os.path.isdir(d)]
    tars = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
    if dirs:
        shutil.copytree(dirs[0], dst); print('Data: copied', dirs[0])
    elif tars:
        shutil.copy(tars[0], DATA)
        with tarfile.open(tars[0]) as t: t.extractall(DATA)
        print('Data: extracted', tars[0])
    else:
        print('Data: downloading CIFAR-10 once before starting the workers...')
        import torchvision
        torchvision.datasets.CIFAR10(DATA, train=True, download=True)
        torchvision.datasets.CIFAR10(DATA, train=False, download=True)

def count_done(d):
    return len(glob.glob(os.path.join(d, '**', 'result.json'), recursive=True))
if os.path.isdir(RESULTS):
    print('Using existing results_exp:', count_done(RESULTS), 'runs')
else:
    cands = [c for c in glob.glob('/kaggle/input/**/results_exp', recursive=True) if os.path.isdir(c)]
    if cands:
        best = max(cands, key=count_done)
        shutil.copytree(best, RESULTS)
        print('Restored from', best, ':', count_done(RESULTS), 'runs')
    else:
        print('Starting from scratch')

In [ ]:
# === Cell 3. Pin the torch + cuDNN versions of the original benchmark (the runner is a subprocess, so no restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'C': ('2.11.0', '0.26.0', 'cu128', 91900),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900),
        'F': ('2.11.0', '0.26.0', 'cu128', 91900),
        'H': ('2.11.0', '0.26.0', 'cu128', 91900),
        'G': ('2.11.0', '0.26.0', 'cu130', 92700)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('torch installed:', have, '| required:', want)
if have != want:
    print('Reinstalling torch to match the original benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, required {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(required', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU: switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: cuDNN could not be matched. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch level differs; results are statistically equivalent but may not be bit-identical.')
else:
    print('Environment matches the original benchmark.')

In [ ]:
# === Cell 4. Multi-GPU runner with a time budget ===
import signal, threading
os.chdir(CODE)
T0 = time.time()
n_gpu = int(subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.device_count())'],
                           capture_output=True, text=True).stdout.strip() or 0)
N = min(N_WORKERS, n_gpu)
if N == 0:
    raise SystemExit('No GPU. Settings -> Accelerator -> GPU T4 x2.')
print('Using', N, 'GPU(s)')

def run_parallel(args, n, budget_s, log):
    os.makedirs(os.path.dirname(log), exist_ok=True)
    f = open(log, 'a'); lk = threading.Lock(); procs = []
    def pump(p, k):
        for line in p.stdout:
            if not line.startswith('['):
                line = f'[w{k}] ' + line
            with lk:
                print(line, end='', flush=True); f.write(line); f.flush()
    for k in range(n):
        e = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), PYTHONUNBUFFERED='1')
        cmd = [sys.executable, 'run_exp.py'] + args + ['--worker', str(k), '--workers', str(n)]
        print(f'$ CUDA_VISIBLE_DEVICES={k}', ' '.join(cmd), flush=True)
        p = subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        th = threading.Thread(target=pump, args=(p, k), daemon=True); th.start()
        procs.append((p, th)); time.sleep(5)
    while any(p.poll() is None for p, _ in procs):
        if time.time() - T0 > budget_s:
            print('\n>>> Time budget reached: stopping the workers (checkpoints are safe).', flush=True)
            for p, _ in procs:
                if p.poll() is None: os.killpg(p.pid, signal.SIGTERM)
            for p, _ in procs:
                try: p.wait(120)
                except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL)
            break
        time.sleep(10)
    rcs = [p.wait() for p, _ in procs]
    for _, th in procs: th.join(5)
    f.close()
    print(f'[exit {rcs}] after {(time.time()-T0)/3600:.2f} h', flush=True)
    return max(rcs, key=abs)

In [ ]:
# === Cell 5. Self-test (first time only) ===
if RUN_CHECKS:
    assert subprocess.call([sys.executable, 'run_exp.py', '--selftest', '--out', '/kaggle/tmp_st']) == 0, 'run_exp SELFTEST failed'

In [ ]:
# === Cell 6. Run ===
subprocess.call([sys.executable, 'run_exp.py', '--plan', '--cells', *CELLS, '--exp', *EXPS])
if not RUN_CHECKS:
    args = ['--exp', *EXPS, '--cells', *CELLS, '--out', RESULTS, '--data', DATA, '--max-hours', str(MAX_HOURS)] + (['--force-env'] if FORCE_ENV else [])
    rc = run_parallel(args, N, TIME_BUDGET_H * 3600, os.path.join(RESULTS, 'kaggle_log.txt'))
    if rc == 2:
        print('!!! Environment/configuration error: see the log above.')

In [ ]:
# === Cell 7. Pilot results (STAGE == 'pilot' only): does ResNet-18 without BN train? ===
import json, glob
if STAGE == 'pilot':
    rows = []
    for f in sorted(glob.glob(f'{RESULTS}/E7p/*/runs/*/result.json')):
        r = json.load(open(f)); h = r['history']
        va = [e.get('val_acc') for e in h]
        pick = lambda ep: f"{100*va[ep-1]:.1f}" if len(va) >= ep and va[ep-1] is not None else '-'
        rows.append((r['cell'], r['lr'], r['diverged'], pick(5), pick(10), pick(20), pick(30), h[-1]['train_loss']))
    print('cell LR      diverged val@5  val@10  val@20  val@30  final train_loss')
    for c, lr, d, a5, a10, a20, a30, tl in rows:
        print(f'{c}  {lr:<8g} {str(d):7}  {a5:>5}  {a10:>6}  {a20:>6}  {a30:>6}  {tl:.3f}')
    best = {c: max([float(a30) for cc, _, d, *_r, a30, _t in rows if cc == c and not d and a30 != '-'] or [0]) for c in ('F', 'H')}
    print('\nbest val@30:', best)
    if best['F'] >= 75:
        print('=> RECOMMENDATION: use cell F (plain ResNet-18 without BN). Stage 2: STAGE="main", USE_H=False.')
    elif best['H'] >= 75:
        print('=> RECOMMENDATION: cell F trains poorly or diverges; use cell H (zero-initialised residual scale). Stage 2: STAGE="main", USE_H=True.')
    else:
        print('=> Both F and H train poorly: stop and review before stage 2.')


In [ ]:
# === Cell 8. Remove checkpoints, show progress, package results ===
if os.path.isdir(RESULTS):
    for res in glob.glob(os.path.join(RESULTS, '**', 'result.json'), recursive=True):
        for ck in Path(os.path.dirname(res)).glob('*.pt*'):
            ck.unlink()
    subprocess.call([sys.executable, 'run_exp.py', '--status', '--cells', *CELLS, '--exp', *EXPS, '--out', RESULTS])
    print('Created', shutil.make_archive(f'{WORK}/results_exp_v5_kaggle_' + STAGE + '', 'zip', WORK, 'results_exp'))
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(DATA, ignore_errors=True)